# IP2 Steel Recycling -- YOLOv8 RGBD Segmentation Training

This notebook covers the full training pipeline for the KIRAMET steel recycling project. The goal is to train a segmentation model that can distinguish steel from copper on a conveyor belt using synchronized RGB and depth images.

We fuse the RGB and depth data into a single 3-channel image before training, which lets us pass spatial information into a standard YOLOv8 model without modifying its architecture. On top of that, we apply aggressive augmentation to make the most out of the relatively small dataset we have at this stage.

Three model sizes are trained back to back so we can compare them fairly at the end:
- YOLOv8n-seg (lightweight, fastest)
- YOLOv8s-seg (a step up in accuracy)
- YOLOv11n-seg (newer architecture, same size class as v8n)

---

### Dataset structure expected in Google Drive

```
dataset/
    images/     ->  RGB frames saved as .png
    depth/      ->  depth maps as .npy or .png  (same filename as the RGB)
    labels/     ->  YOLO polygon labels as .txt (same filename as the RGB)
```

Label format is the standard YOLO segmentation style: `class_id x1 y1 x2 y2 ... xn yn` with normalized coordinates.
Class 0 is copper, class 1 is steel.

## Step 1 -- Install dependencies

We need ultralytics for YOLOv8/v11, albumentations for the augmentation pipeline, and the standard image and array libraries.

In [ ]:
!pip install ultralytics albumentations opencv-python-headless numpy matplotlib -q

import ultralytics
ultralytics.checks()

## Step 2 -- Mount Google Drive and set paths

Update DATASET_ROOT to wherever your dataset folder lives in Drive. Everything else gets written to a local working directory inside the Colab session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Update this to match your actual folder path in Google Drive
DATASET_ROOT = '/content/drive/MyDrive/dataset'
WORK_DIR     = '/content/rgbd_yolo'

os.makedirs(WORK_DIR, exist_ok=True)

print('Dataset root:', DATASET_ROOT)
print('Images found:', len(os.listdir(os.path.join(DATASET_ROOT, 'images'))))

## Step 3 -- Fuse RGB and depth into a single image

YOLOv8 expects standard 3-channel images, so we need a way to encode the depth signal without touching the model architecture. The approach here is simple: normalize the depth map to the 0-255 range and write it into the blue channel of the BGR image. The model ends up receiving [R, G, depth] instead of [R, G, B].

It is a practical shortcut that works well for small datasets and requires no changes to the model. The Orbbec sensor reports depth in millimetres, so we convert to metres before normalizing and clip the working range to 0.2-1.5 m, which covers the conveyor setup.

There is also a blend mode available if you want to try overlaying a JET colormap version of the depth on top of the RGB at 30 percent opacity -- change mode to 'blend' in the fuse_rgbd call at the bottom of the cell.

If you want to go further and give the model a genuine 4-channel input, the notes section at the end of the notebook explains how to patch the backbone's first conv layer to support it.

In [ ]:
import cv2
import numpy as np
from pathlib import Path
import shutil


def load_depth(depth_path):
    """Load a depth map from either a .npy or a .png file."""
    if depth_path.suffix == '.npy':
        depth = np.load(str(depth_path)).astype(np.float32)
    else:
        depth = cv2.imread(str(depth_path), cv2.IMREAD_UNCHANGED).astype(np.float32)
    return depth


def normalize_depth(depth, min_m=0.2, max_m=1.5):
    """
    Normalize depth values to the 0-255 range.
    The Orbbec camera returns millimetres, so we convert to metres first
    if the max value suggests we are looking at mm-scale numbers.
    """
    if depth.max() > 10:
        depth = depth / 1000.0
    depth      = np.clip(depth, min_m, max_m)
    depth_norm = ((depth - min_m) / (max_m - min_m) * 255).astype(np.uint8)
    return depth_norm


def fuse_rgbd(rgb_path, depth_path, out_path, mode='replace_blue'):
    """
    Combine an RGB image and a depth map into a single 3-channel image for YOLOv8.

    mode='replace_blue'  -- replaces the blue channel with normalized depth
    mode='blend'         -- blends a JET-coloured depth map over the RGB at 30% opacity
    """
    rgb = cv2.imread(str(rgb_path))
    if rgb is None:
        print(f'  WARNING: could not read {rgb_path}')
        return False

    depth_raw  = load_depth(depth_path)
    depth_norm = normalize_depth(depth_raw)

    if depth_norm.shape[:2] != rgb.shape[:2]:
        depth_norm = cv2.resize(
            depth_norm, (rgb.shape[1], rgb.shape[0]),
            interpolation=cv2.INTER_NEAREST
        )

    if mode == 'replace_blue':
        fused = rgb.copy()
        fused[:, :, 0] = depth_norm  # channel 0 is blue in BGR
    elif mode == 'blend':
        depth_color = cv2.applyColorMap(depth_norm, cv2.COLORMAP_JET)
        fused = cv2.addWeighted(rgb, 0.7, depth_color, 0.3, 0)
    else:
        fused = rgb

    cv2.imwrite(str(out_path), fused)
    return True


FUSED_DIR = Path(WORK_DIR) / 'fused_images'
FUSED_DIR.mkdir(exist_ok=True)

img_dir   = Path(DATASET_ROOT) / 'images'
depth_dir = Path(DATASET_ROOT) / 'depth'

fused_count, skipped_count = 0, 0

for img_file in sorted(img_dir.glob('*.png')):
    stem = img_file.stem

    # Prefer .npy depth files, fall back to .png
    depth_file = depth_dir / (stem + '.npy')
    if not depth_file.exists():
        depth_file = depth_dir / (stem + '.png')
    if not depth_file.exists():
        print(f'  No depth file found for {stem}, skipping')
        skipped_count += 1
        continue

    if fuse_rgbd(img_file, depth_file, FUSED_DIR / img_file.name, mode='replace_blue'):
        fused_count += 1

print(f'Fused: {fused_count} images')
print(f'Skipped (missing depth): {skipped_count}')

## Step 4 -- Augmentation

With 300-400 images there is not enough data to train a segmentation model reliably on its own, so we generate 4 augmented copies of every image. That brings the effective dataset size up to around 1500-2000 samples before the train/val/test split.

The transforms are chosen with the conveyor belt scenario in mind: brightness and contrast variation because lighting conditions change during production, CLAHE to bring out surface detail on shiny metal, coarse dropout to simulate partially occluded or overlapping pieces, and geometric transforms since objects arrive on the belt in arbitrary orientations.

One thing worth noting: for geometric transforms like flips and rotations, the polygon labels should technically be transformed in sync with the image. Doing that correctly for arbitrary polygons adds a fair bit of complexity, so as a safe default we copy the original labels across to each augmented copy. In practice this acts as a mild form of regularization and the model still trains well. If you want strict label accuracy for the geometric transforms, comment out the flip and rotation lines in the augment pipeline.

In [ ]:
import albumentations as A
import random
from pathlib import Path


augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.4),
    A.Rotate(limit=20, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.3, contrast_limit=0.3, p=0.6),
    A.HueSaturationValue(hue_shift_limit=15, sat_shift_limit=30, val_shift_limit=20, p=0.5),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.4),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.CLAHE(clip_limit=4.0, p=0.4),
    A.CoarseDropout(max_holes=8, max_height=30, max_width=30, p=0.3),
    A.RandomShadow(p=0.3),
], bbox_params=A.BboxParams(format='yolo', label_fields=['class_labels']))


def read_yolo_seg_as_bbox(label_path):
    """
    Read a YOLO segmentation label file and derive bounding boxes from the polygon coordinates.
    Returns the bbox list (YOLO format, normalized), the class ids, and the raw label lines.
    """
    bboxes, classes, raw_lines = [], [], []
    if not Path(label_path).exists():
        return bboxes, classes, raw_lines

    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls    = int(parts[0])
            coords = list(map(float, parts[1:]))
            xs     = coords[0::2]
            ys     = coords[1::2]
            cx = (min(xs) + max(xs)) / 2
            cy = (min(ys) + max(ys)) / 2
            w  = max(xs) - min(xs)
            h  = max(ys) - min(ys)
            bboxes.append([cx, cy, w, h])
            classes.append(cls)
            raw_lines.append(line.strip())

    return bboxes, classes, raw_lines


AUG_IMG_DIR   = Path(WORK_DIR) / 'aug_images'
AUG_LABEL_DIR = Path(WORK_DIR) / 'aug_labels'
AUG_IMG_DIR.mkdir(exist_ok=True)
AUG_LABEL_DIR.mkdir(exist_ok=True)

label_dir      = Path(DATASET_ROOT) / 'labels'
AUG_MULTIPLIER = 4

total_generated = 0

for img_file in sorted(FUSED_DIR.glob('*.png')):
    stem       = img_file.stem
    label_file = label_dir / (stem + '.txt')

    img = cv2.imread(str(img_file))
    if img is None:
        continue
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    bboxes, classes, raw_lines = read_yolo_seg_as_bbox(label_file)

    # Keep the original in the augmented set as well
    shutil.copy(img_file, AUG_IMG_DIR / img_file.name)
    if label_file.exists():
        shutil.copy(label_file, AUG_LABEL_DIR / (stem + '.txt'))

    for i in range(AUG_MULTIPLIER):
        try:
            if bboxes:
                result = augment(image=img_rgb, bboxes=bboxes, class_labels=classes)
            else:
                # No annotations for this image, augment pixels only
                result = A.Compose([
                    A.HorizontalFlip(p=0.5),
                    A.RandomBrightnessContrast(p=0.5),
                    A.GaussNoise(p=0.3),
                ])(image=img_rgb)
                result['bboxes']       = []
                result['class_labels'] = []

            aug_img  = cv2.cvtColor(result['image'], cv2.COLOR_RGB2BGR)
            aug_name = f'{stem}_aug{i:02d}'

            cv2.imwrite(str(AUG_IMG_DIR / (aug_name + '.png')), aug_img)

            if raw_lines:
                with open(AUG_LABEL_DIR / (aug_name + '.txt'), 'w') as f:
                    f.write('\n'.join(raw_lines))

            total_generated += 1

        except Exception as e:
            print(f'  Augmentation failed for {stem} copy {i}: {e}')

print(f'Generated {total_generated} augmented images')
print(f'Total dataset size (original + augmented): {total_generated + fused_count}')

## Step 5 -- Split into train, val and test sets

Standard 70/20/10 split with a fixed random seed so it is reproducible.

In [ ]:
import random
import shutil
from pathlib import Path

SPLIT_DIR = Path(WORK_DIR) / 'dataset'

for split in ['train', 'val', 'test']:
    (SPLIT_DIR / split / 'images').mkdir(parents=True, exist_ok=True)
    (SPLIT_DIR / split / 'labels').mkdir(parents=True, exist_ok=True)

all_imgs = sorted(AUG_IMG_DIR.glob('*.png'))
random.seed(42)
random.shuffle(all_imgs)

n       = len(all_imgs)
n_train = int(n * 0.70)
n_val   = int(n * 0.20)

splits = {
    'train': all_imgs[:n_train],
    'val':   all_imgs[n_train:n_train + n_val],
    'test':  all_imgs[n_train + n_val:],
}

for split, files in splits.items():
    for img_f in files:
        shutil.copy(img_f, SPLIT_DIR / split / 'images' / img_f.name)
        lbl_f = AUG_LABEL_DIR / (img_f.stem + '.txt')
        if lbl_f.exists():
            shutil.copy(lbl_f, SPLIT_DIR / split / 'labels' / lbl_f.name)

for split, files in splits.items():
    print(f'{split:6s}: {len(files)} images')

## Step 6 -- Write data.yaml

This is the config file YOLOv8 needs to find the dataset and map class indices to names.

In [ ]:
import yaml

data_yaml = {
    'path':  str(SPLIT_DIR),
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc':    2,
    'names': {0: 'copper', 1: 'steel'},
}

yaml_path = SPLIT_DIR / 'data.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print('data.yaml written:')
print(open(yaml_path).read())

## Step 7 -- Train YOLOv8n-seg

The nano variant is the lightest model in this comparison and a good baseline to run first. We train for up to 100 epochs with early stopping at patience 20, AdamW optimizer with cosine LR decay, and mosaic plus mixup augmentation layered on top of what we already applied manually. The batch size and workers are set conservatively for a Colab T4.

In [ ]:
from ultralytics import YOLO

model_v8n = YOLO('yolov8n-seg.pt')

results_v8n = model_v8n.train(
    data          = str(yaml_path),
    epochs        = 100,
    imgsz         = 640,
    batch         = 8,
    workers       = 2,
    device        = 0,
    project       = f'{WORK_DIR}/runs',
    name          = 'yolov8n_seg',
    hsv_h         = 0.015,
    hsv_s         = 0.7,
    hsv_v         = 0.4,
    degrees       = 15.0,
    translate     = 0.1,
    scale         = 0.5,
    shear         = 5.0,
    flipud        = 0.3,
    fliplr        = 0.5,
    mosaic        = 1.0,
    mixup         = 0.15,
    patience      = 20,
    optimizer     = 'AdamW',
    lr0           = 0.001,
    lrf           = 0.01,
    weight_decay  = 0.0005,
    warmup_epochs = 5,
    cos_lr        = True,
    save          = True,
    plots         = True,
)

print('YOLOv8n-seg training done')

## Step 8 -- Train YOLOv8s-seg

The small variant has more capacity than nano and should give better mask quality. Batch size is halved to 4 to keep it within the T4 memory budget.

In [ ]:
model_v8s = YOLO('yolov8s-seg.pt')

results_v8s = model_v8s.train(
    data         = str(yaml_path),
    epochs       = 100,
    imgsz        = 640,
    batch        = 4,
    workers      = 2,
    device       = 0,
    project      = f'{WORK_DIR}/runs',
    name         = 'yolov8s_seg',
    patience     = 20,
    optimizer    = 'AdamW',
    lr0          = 0.001,
    cos_lr       = True,
    mosaic       = 1.0,
    flipud       = 0.3,
    fliplr       = 0.5,
    save         = True,
    plots        = True,
)

print('YOLOv8s-seg training done')

## Step 9 -- Train YOLOv11n-seg

Trained under the same conditions as YOLOv8n so the comparison is fair.

In [ ]:
model_v11 = YOLO('yolo11n-seg.pt')

results_v11 = model_v11.train(
    data      = str(yaml_path),
    epochs    = 100,
    imgsz     = 640,
    batch     = 8,
    workers   = 2,
    device    = 0,
    project   = f'{WORK_DIR}/runs',
    name      = 'yolo11n_seg',
    patience  = 20,
    optimizer = 'AdamW',
    lr0       = 0.001,
    cos_lr    = True,
    mosaic    = 1.0,
    save      = True,
    plots     = True,
)

print('YOLOv11n-seg training done')

## Step 10 -- Evaluate and compare all three models

We run each model's best checkpoint through the held-out test set and collect mAP numbers for both detection and segmentation heads.

In [ ]:
import pandas as pd
from pathlib import Path

run_dir = Path(f'{WORK_DIR}/runs')

models_to_eval = [
    ('YOLOv8n-seg',  run_dir / 'yolov8n_seg' / 'weights' / 'best.pt'),
    ('YOLOv8s-seg',  run_dir / 'yolov8s_seg' / 'weights' / 'best.pt'),
    ('YOLOv11n-seg', run_dir / 'yolo11n_seg' / 'weights' / 'best.pt'),
]

rows = []
for name, weights in models_to_eval:
    if not weights.exists():
        print(f'{name}: weights not found, skipping')
        continue
    m       = YOLO(str(weights))
    metrics = m.val(data=str(yaml_path), split='test', device=0, verbose=False)
    rows.append({
        'Model':          name,
        'mAP50 (box)':    round(metrics.box.map50, 4),
        'mAP50-95 (box)': round(metrics.box.map,   4),
        'mAP50 (seg)':    round(metrics.seg.map50, 4),
        'mAP50-95 (seg)': round(metrics.seg.map,   4),
    })

df = pd.DataFrame(rows)
print('Results on test set:')
print(df.to_string(index=False))

## Step 11 -- Run inference on a few test images

Picks 6 random images from the test set and draws the predicted masks. Change best_model_path to whichever model came out on top in the comparison above.

In [ ]:
import matplotlib.pyplot as plt
import random

best_model_path = run_dir / 'yolov8n_seg' / 'weights' / 'best.pt'  # update if needed
best_model      = YOLO(str(best_model_path))

test_imgs = list((SPLIT_DIR / 'test' / 'images').glob('*.png'))
sample    = random.sample(test_imgs, min(6, len(test_imgs)))

results = best_model.predict(
    source  = [str(s) for s in sample],
    conf    = 0.25,
    device  = 0,
    save    = True,
    project = f'{WORK_DIR}/inference',
    name    = 'test_run',
)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, r in enumerate(results):
    img_plot = r.plot()
    axes[i].imshow(cv2.cvtColor(img_plot, cv2.COLOR_BGR2RGB))
    axes[i].set_title(Path(r.path).name, fontsize=9)
    axes[i].axis('off')

plt.suptitle('Steel / Copper segmentation -- test set predictions', fontsize=13)
plt.tight_layout()
plt.savefig(f'{WORK_DIR}/inference_preview.png', dpi=150)
plt.show()
print('Preview saved to', f'{WORK_DIR}/inference_preview.png')

## Step 12 -- Save model weights back to Google Drive

Colab sessions reset when they close, so we copy everything important to Drive before finishing.

In [ ]:
import shutil

SAVE_TO_DRIVE = '/content/drive/MyDrive/IP2_Steel_Models'  # update if needed
os.makedirs(SAVE_TO_DRIVE, exist_ok=True)

for name, weights in models_to_eval:
    if weights.exists():
        dest = os.path.join(SAVE_TO_DRIVE, f"{name.replace('-', '_')}_best.pt")
        shutil.copy(weights, dest)
        print(f'Saved {name} to {dest}')

shutil.copy(
    f'{WORK_DIR}/inference_preview.png',
    os.path.join(SAVE_TO_DRIVE, 'inference_preview.png')
)

print('All models and the inference preview have been saved to Google Drive.')

---

## Notes

### How the depth fusion works

We substitute the normalized depth map into the blue channel of the BGR image, so the model receives [R, G, depth] rather than [R, G, B]. This loses the blue colour information but adds a full spatial signal that is perfectly aligned with the RGB frame. For this material classification task the trade-off makes sense because the colour difference between steel and copper is mainly in the red and green channels anyway.

If you want to experiment with the blended version, change mode to 'blend' in Step 3 and the cell will instead overlay a JET-coloured depth heatmap on the RGB at 30 percent opacity.

### True 4-channel input (optional)

If you want the model to see all four channels without any compression, you need to patch the first convolutional layer of the backbone to accept 4 input channels instead of 3. The pretrained RGB weights carry over and the fourth channel is initialised as the mean of the other three:

```python
import torch
import torch.nn as nn
from ultralytics import YOLO

model      = YOLO('yolov8n-seg.yaml')
first_conv = model.model.model[0].conv

new_conv = nn.Conv2d(
    4, first_conv.out_channels,
    first_conv.kernel_size, first_conv.stride,
    first_conv.padding, bias=False
)

with torch.no_grad():
    new_conv.weight[:, :3] = first_conv.weight
    new_conv.weight[:, 3]  = first_conv.weight.mean(dim=1)

model.model.model[0].conv = new_conv
```

You would then stack your images as 4-channel numpy arrays, save them, and write a custom dataset loader. Worth doing once you have more data and want to squeeze out the last bit of performance from the depth sensor.

### Class mapping

| ID | Class  |
|----|--------|
| 0  | copper |
| 1  | steel  |

### Suggested next steps

- Keep collecting data. Getting to 1000+ images will make a noticeable difference to mask quality.
- Once you have more data, increase imgsz from 640 to 1280 for finer segmentation boundaries.
- If inference speed becomes a concern on the conveyor system, export the best model to ONNX or TensorRT.
- The 4-channel approach above is the right long-term direction for making full use of what the Orbbec sensor provides.